# 04 · Cleaning — Spend

**Goal:** clean the Spend table — fix the date type, run data checks, remove duplicates, and confirm the total ad budget.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load & inspect

In [2]:
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')

spend = pd.read_excel(
    DATA_RAW / 'Spend (Done).xlsx')
spend['Date'] = pd.to_datetime(spend['Date'], format='%d.%m.%Y')

In [3]:
spend.shape

(20779, 8)

In [4]:
spend.head()

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-07-03,Facebook Ads,NaN,0,0.00,0,NaN,NaN
3,2023-07-03,Google Ads,NaN,0,0.00,0,NaN,NaN
4,2023-07-03,CRM,NaN,0,0.00,0,NaN,NaN


In [5]:
spend.dtypes

Date           datetime64[ns]
Source                 object
Campaign               object
Impressions             int64
Spend                 float64
Clicks                  int64
AdGroup                object
Ad                     object
dtype: object

In [6]:
spend.isna().sum()

Date              0
Source            0
Campaign       5994
Impressions       0
Spend             0
Clicks            0
AdGroup        6828
Ad             6828
dtype: int64

In [7]:
spend.describe() 

,Date,Impressions,Spend,Clicks
count,20779,20779.000000,20779.000000,20779.000000
mean,2024-01-14 22:32:40.864334080,2458.203475,7.195892,23.990616
min,2023-07-03 00:00:00,0.000000,0.000000,0.000000
25%,2023-10-13 00:00:00,0.000000,0.000000,0.000000
50%,2024-01-27 00:00:00,63.000000,0.580000,1.000000
75%,2024-04-16 00:00:00,709.000000,5.750000,12.000000
max,2024-06-21 00:00:00,431445.000000,774.000000,2415.000000
std,NaN,11442.528075,26.760080,85.245714


In [8]:
spend.columns.tolist()

['Date',
 'Source',
 'Campaign',
 'Impressions',
 'Spend',
 'Clicks',
 'AdGroup',
 'Ad']

In [9]:
categorical_cols = ['Date',
 'Source',
 'Campaign',
 'Impressions',
 'Spend',
 'Clicks',
 'AdGroup',
 'Ad']

for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(spend[col].value_counts(dropna=False))


=== Date ===
Date
2024-04-25    100
2024-04-15    100
2024-04-13     99
2024-04-16     99
2024-04-17     98
             ... 
2023-07-12     29
2023-07-11     28
2023-07-04     21
2023-07-05     21
2023-07-03     17
Name: count, Length: 355, dtype: int64

=== Source ===
Source
Facebook Ads      9732
Tiktok Ads        3066
Youtube Ads       1926
Google Ads        1428
Telegram posts    1003
Bloggers           787
Webinar            766
SMM                614
Organic            518
CRM                355
Test               262
Partnership        234
Offline             61
Radio               27
Name: count, dtype: int64

=== Campaign ===
Campaign
NaN                            5994
12.07.2023wide_DE              2073
02.07.23wide_DE                1685
04.07.23recentlymoved_DE       1398
youtube_shorts_DE              1223
07.07.23LAL_DE                 1181
03.07.23women                  1171
12.09.23interests_Uxui_DE      1143
15.07.23b_DE                    529
24.09.23retargeting_DE

## 2. Data checks

In [10]:
print("Source for rows without Campaign:")
print(spend[spend['Campaign'].isna()]['Source'].value_counts())

print("\nSource for rows without AdGroup:")
print(spend[spend['AdGroup'].isna()]['Source'].value_counts())

Source for rows without Campaign:
Source
Telegram posts    1003
Bloggers           787
SMM                614
Google Ads         519
Organic            518
Facebook Ads       518
Youtube Ads        497
Tiktok Ads         436
CRM                355
Webinar            332
Partnership        234
Test                93
Offline             61
Radio               27
Name: count, dtype: int64

Source for rows without AdGroup:
Source
Google Ads        1428
Telegram posts    1003
Bloggers           787
SMM                543
Organic            518
Facebook Ads       518
Youtube Ads        497
Tiktok Ads         436
CRM                355
Webinar            328
Partnership        234
Test                93
Offline             61
Radio               27
Name: count, dtype: int64


In [11]:
test_spend = spend[spend['Source'] == 'Test']
print(f"Test rows: {len(test_spend)}")
print(f"Spend sum: {test_spend['Spend'].sum():.2f} EUR")
print(f"Impressions: {test_spend['Impressions'].sum()}")
print(f"Clicks: {test_spend['Clicks'].sum()}")

Test rows: 262
Spend sum: 608.21 EUR
Impressions: 43969
Clicks: 1226


In [12]:
print("Spend = 0 by source:")
print(spend[spend['Spend'] == 0]['Source'].value_counts())

Spend = 0 by source:
Source
Tiktok Ads        868
Facebook Ads      844
Telegram posts    793
Bloggers          712
Google Ads        523
SMM               518
Organic           518
Youtube Ads       505
Webinar           462
CRM               355
Partnership       234
Test               97
Offline            61
Radio              21
Name: count, dtype: int64


In [13]:
print(f"Data period: from {spend['Date'].min()} to {spend['Date'].max()}")
print(f"Days covered: {spend['Date'].nunique()}")
print(f"Total budget: {spend['Spend'].sum():.0f} EUR")

Data period: from 2023-07-03 00:00:00 to 2024-06-21 00:00:00
Days covered: 355
Total budget: 149523 EUR


In [14]:
print(f"Fully duplicated rows: {spend.duplicated().sum()}")

Fully duplicated rows: 917


### 3. Duplicates

Look at the duplicates.

In [15]:
# Which rows are duplicated
duplicates = spend[spend.duplicated(keep=False)].sort_values(['Date', 'Source', 'Campaign'])
print(f"Total duplicate rows (incl. originals): {len(duplicates)}")
print(f"\nFirst 20:")
print(duplicates.head(20))

Total duplicate rows (incl. originals): 1527

First 20:
           Date    Source Campaign  Impressions  Spend  Clicks AdGroup   Ad
753  2023-07-23  Bloggers      NaN            0    0.0       0     NaN  NaN
755  2023-07-23  Bloggers      NaN            0    0.0       0     NaN  NaN
768  2023-07-24  Bloggers      NaN            0    0.0       0     NaN  NaN
789  2023-07-24  Bloggers      NaN            0    0.0       0     NaN  NaN
841  2023-07-25  Bloggers      NaN            0    0.0       0     NaN  NaN
844  2023-07-25  Bloggers      NaN            0    0.0       0     NaN  NaN
895  2023-07-26  Bloggers      NaN            0    0.0       0     NaN  NaN
899  2023-07-26  Bloggers      NaN            0    0.0       0     NaN  NaN
950  2023-07-27  Bloggers      NaN            0    0.0       0     NaN  NaN
958  2023-07-27  Bloggers      NaN            0    0.0       0     NaN  NaN
973  2023-07-28  Bloggers      NaN            0    0.0       0     NaN  NaN
977  2023-07-28  Bloggers      N

In [16]:
print(f"Before removal: {spend.shape}")
spend = spend.drop_duplicates()
print(f"After removal: {spend.shape}")

Before removal: (20779, 8)
After removal: (19862, 8)


### Removing duplicates

Found 917 fully duplicated rows. All are empty — zeros in `Impressions`, `Spend`, `Clicks`. Since these are not transactions and have no unique ID, they carry no information for us.

## 4. Cleaning summary — Spend

**Period:** 03.07.2023 — 21.06.2024 (355 days covered)
**Acquisition channels:** 14

> Note: Spend has 14 sources vs 13 in Deals — Spend includes Radio, which is absent in Deals.

**Type conversion**
- `Date` → datetime
- Other types inferred correctly (Spend as float, Impressions/Clicks as int)

**Source `Test`:** kept.

*Note: Spend analysis is an additional branch — the assignment scope covered conversion by source, not spend-side analysis.*

In [17]:
spend.to_parquet(DATA_PROCESSED / 'spend_clean.parquet')
spend.to_excel(DATA_PROCESSED / 'spend_clean.xlsx', index=False)